# analysis

## re-cluster

In [ ]:
# Load required modules
import snapatac2 as snap
import scanpy as sc
import scanpy.external as sce

In [ ]:
data = sc.read_h5ad('./processed/adata.h5ad')
data

In [ ]:
data.obs['final_ann'] = [item.replace(" ", "_") for item in data.obs['final_ann'].tolist()]
cell = list(set(data.obs['final_ann'].tolist()))
cell

In [ ]:
# Adjust n_pcs for cell types with few cells
# HSC/MPP: saving may raise an error
# for i in range(len(cell)):
#     print(i)
#     print(cell[i])
#     rna = data[data.obs['final_ann'] == cell[i]]
#     sc.pp.normalize_total(rna, target_sum=1e4)
#     sc.pp.log1p(rna)
#     sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
#     rna = rna[:, rna.var.highly_variable]
#     snap.tl.spectral(rna, features=None)
#     rna.obsm['X_pca'] = rna.obsm['X_spectral']
#     sce.pp.harmony_integrate(rna, 'sample')
#     rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
#     sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
#     sc.tl.umap(rna)
#     rna.write_h5ad('./processed/subcluster/'+cell[i]+'.h5ad')

In [ ]:
    rna = data[data.obs['final_ann'] == 'ILC']
    sc.pp.normalize_total(rna, target_sum=1e4)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
    rna = rna[:, rna.var.highly_variable]
    snap.tl.spectral(rna, features=None)
    rna.obsm['X_pca'] = rna.obsm['X_spectral']
    sce.pp.harmony_integrate(rna, 'sample')
    rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
    sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
    sc.tl.umap(rna)
    rna.write_h5ad('./processed/subcluster/'+'ILC'+'.h5ad')

In [ ]:
rna = sc.read_h5ad('./processed/subcluster/T_cells.h5ad')
rna.obs['final_ann'].value_counts()

In [ ]:
# import snapatac2 as snap
# import scanpy as sc
# import scanpy.external as sce

# data = sc.read_h5ad('./processed/adata.h5ad')
# data.obs['final_ann'] = [item.replace(" ", "_") for item in data.obs['final_ann'].tolist()]
# cell = list(set(data.obs['final_ann'].tolist()))

# for i in range(len(cell)):
#     print(i)
#     print(cell[i])
#     rna = data[data.obs['final_ann'] == cell[i]]
#     sc.pp.normalize_total(rna, target_sum=1e4)
#     sc.pp.log1p(rna)
#     sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
#     rna = rna[:, rna.var.highly_variable]
#     snap.tl.spectral(rna, features=None)
#     rna.obsm['X_pca'] = rna.obsm['X_spectral']
#     sce.pp.harmony_integrate(rna, 'sample')
#     rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
#     sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
#     sc.tl.umap(rna)
#     rna.write_h5ad('./processed/subcluster/'+cell[i]+'.h5ad')

## demo for cell annotation

In [ ]:
# Compute
rna = sc.read_h5ad('/path/to/annotation_project/subset_ann/B_cells.h5ad')
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
rna = rna[:, rna.var.highly_variable]
snap.tl.spectral(rna, features=None)
rna.obsm['X_pca'] = rna.obsm['X_spectral']
sce.pp.harmony_integrate(rna, 'sample')
rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
sc.tl.umap(rna)

In [ ]:
# Changing resolution changes the number of clusters
sc.tl.leiden(rna, flavor="igraph", n_iterations=2, resolution=0.1) 

In [ ]:
rna

In [ ]:
# Visualize Leiden clusters alongside cellType_raw_level3, cellType_celltypist_level3, and cellType_singlecellnet_level3
sc.pl.umap(rna, color='leiden')

In [ ]:
sc.pl.dotplot(rna, ['ITGAX', 'CD27',  'IGHD', 'TCL1A'], groupby='leiden')

## Confirm details of incorrectly annotated cells

In [ ]:
import scanpy as sc

In [ ]:
data = sc.read('./processed/subcluster/raw/Mast_cells.h5ad')
data

In [ ]:
data.obs['cellType_raw_level2'].value_counts()

In [ ]:
mac = sc.read('./processed/subcluster/raw/Macrophages.h5ad')
mac

In [ ]:
mac.obs['cellType_raw_level2'].value_counts()

In [ ]:
bl = sc.read('./processed/subcluster/raw/B-cell_lineage.h5ad')
bl

In [ ]:
bl.obs['cellType_raw_level2'].value_counts()

## Draw a stacked column chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

# Configure the plot font
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']  # Use a portable sans-serif font
matplotlib.rcParams['axes.unicode_minus'] = False  # Configure minus-sign rendering

# Prepare data
data = {
    'Cell Type': ['T cells', 'notAvailable', 'HSC/MPP', 'Monocytes', 'ILC', 'B cells', 'Erythroid', 
                  'Early MK', 'DC', 'B-cell lineage', 'pDC', 'Plasma cells', 'Cycling cells', 
                  'Macrophages', 'Mast cells'],
    'Color': ['#ff9896', '#cccecf', '#e377c2', '#ffbb78', '#b5bd61', '#1f77b4', '#8c564b', '#aa40fc',
              '#d62728', '#ff7f0e', '#c5b0d5', '#98df8a', '#279e68', '#17becf', '#aec7e8'],
    'Mast cells': [9, 0, 1202, 696, 20, 2, 354, 20, 6, 8, 1, 0, 0, 1, 507],
    'Macrophages': [12, 0, 66, 170, 2, 5, 19, 4, 41, 0, 0, 1, 0, 19, 0],
    'B-cell lineage': [0, 0, 178, 0, 0, 8, 1, 0, 0, 260, 1, 0, 0, 0, 0]
}

df = pd.DataFrame(data)

# Calculate proportions
groups = ['Mast cells', 'Macrophages', 'B-cell lineage']
for group in groups:
    df[f'{group}_prop'] = df[group] / df[group].sum() * 100

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

# Set bar widths and positions for each group
bar_width = 0.25
x_positions = [0, 1, 2]

# Draw stacked bars for each group
for i, group in enumerate(groups):
    # Sort proportions for stacking
    temp_df = df[['Cell Type', 'Color', f'{group}_prop']].copy()
    temp_df = temp_df[temp_df[f'{group}_prop'] > 0].sort_values(f'{group}_prop', ascending=True)  # Sort in ascending order for bottom-up stacking
    
    bottom = 0  # Initialize the bottom of the stack
    for _, row in temp_df.iterrows():
        ax.bar(x_positions[i], row[f'{group}_prop'], bar_width, bottom=bottom, 
               color=row['Color'], label=row['Cell Type'] if i == 0 else None)
        bottom += row[f'{group}_prop']

# Set plot properties
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Mast cell group', 'Macrophage group', 'B-cell lineage group'])
ax.set_ylabel('Proportion (%)')
ax.set_ylim(0, 100)
ax.set_title('Cell-type proportions by group (ordered within each group)')

# Show each legend label only once
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))  # Remove duplicate labels
ax.legend(by_label.values(), by_label.keys(), loc='center left', bbox_to_anchor=(1, 0.5))

# Adjust the layout to accommodate the legend
plt.tight_layout()

# Save the plot
plt.savefig('stacked_bar_proportion_sorted.pdf', dpi=300, bbox_inches='tight')
plt.close()

# visual

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np

## Organize subgroup annotations

In [ ]:
lis = os.listdir('./celltype_clustered/')[:11]
lis

In [ ]:
subcluster = sc.read_h5ad('./celltype_clustered/'+lis[0]).obs[['leiden']]
subcluster

In [ ]:
for i in range(1,11):
    print(i)
    sub = sc.read_h5ad('./celltype_clustered/'+lis[i]).obs[['leiden']]
    subcluster = pd.concat([subcluster, sub])

In [ ]:
subcluster

In [ ]:
subcluster['leiden'].value_counts()

## merge

In [ ]:
data = sc.read_h5ad('/path/to/hemelas_project/z_others/8_new/processed/adata.h5ad')
data

In [ ]:
# 42 types
# ann = data.obs[['final_ann']]
# dif = list(set(ann.index.tolist()) - set(subcluster.index.tolist()))
# ann_dif = ann.loc[dif,:]
# ann_dif.columns = ['leiden']
# all = pd.concat([subcluster,ann_dif])
# all_sort = all.loc[data.obs_names.tolist(),:]
# all_sort.to_csv('./subcluser.csv')

In [ ]:
all_sort = pd.read_csv('./subcluser.csv', index_col=0)
all_sort.replace('Type1 helper T cells', 'Type 1 helper T cells', inplace=True)

In [ ]:
data.obs['ann_level3'] = all_sort

In [ ]:
# cell level
cell = pd.read_csv('/path/to/hemelas_project/z_others/6_ann/origin/sc_105.csv')
cell = cell.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
cell

In [ ]:
result = dict(zip(cell['level_3'], cell['level_1']))
result

In [ ]:
all_level1 = all_sort.replace(result)
all_level1

In [ ]:
all_level1['leiden'].value_counts()

In [ ]:
result = dict(zip(cell['level_2'], cell['level_1']))
result

In [ ]:
all_level1 = data.obs[['final_ann']].replace(result)
all_level1.value_counts()

In [ ]:
data

In [ ]:
data.obs['ann_level1'] = all_level1

In [ ]:
data

In [ ]:
data.obs[['ann_level1', 'final_ann', 'ann_level3']]

## Sankey diagram

In [ ]:
df = data.obs[['ann_level1', 'final_ann', 'ann_level3']].copy()
df.columns = ['l1', 'l2', 'l3']
df

In [ ]:
df.to_csv('./celltype_l1_l3.csv')

In [ ]:
df['l1'] = df['l1'].astype(str) +'l1'
df['l2'] = df['l2'].astype(str) +'l2'
df['l3'] = df['l3'].astype(str) +'l3'

In [ ]:
import plotly.graph_objects as go

# Step 1: count cells for l1-to-l2 transitions
flow_l1_l2 = df.groupby(['l1', 'l2']).size().reset_index(name='count')
# Step 2: count cells for l2-to-l3 transitions
flow_l2_l3 = df.groupby(['l2', 'l3']).size().reset_index(name='count')

# Combine flow counts
flows = pd.concat([flow_l1_l2, flow_l2_l3], axis=0)

In [ ]:
# Extract unique nodes in hierarchical order
nodes_l1 = df['l1'].unique().tolist()
nodes_l2 = df['l2'].unique().tolist()
nodes_l3 = df['l3'].unique().tolist()
all_nodes = nodes_l1 + nodes_l2 + nodes_l3

# Assign a unique index to each node
node_indices = {node: i for i, node in enumerate(all_nodes)}

# Build links (source, target, value)
links = []
# Add l1-to-l2 links
for _, row in flow_l1_l2.iterrows():
    source = node_indices[row['l1']]
    target = node_indices[row['l2']]
    links.append({'source': source, 'target': target, 'value': row['count']})
# Add l2-to-l3 links
for _, row in flow_l2_l3.iterrows():
    source = node_indices[row['l2']]
    target = node_indices[row['l3']]
    links.append({'source': source, 'target': target, 'value': row['count']})

# Convert to the format required for a Sankey diagram
sankey_links = {
    'source': [link['source'] for link in links],
    'target': [link['target'] for link in links],
    'value': [link['value'] for link in links]
}

In [ ]:
# Create the Sankey diagram
fig = go.Figure(go.Sankey(
    node={
        'label': all_nodes,              # Node labels
        'pad': 20,                       # Node spacing
        'thickness': 30,                 # Node thickness
        'line': {'color': 'black', 'width': 0},  # Node borders
        'color': 'lightblue'             # Node colors
    },
    link={
        'source': sankey_links['source'],  # Source node indices
        'target': sankey_links['target'],  # Target node indices
        'value': sankey_links['value'],    # Flow values
        'color': 'rgba(150, 150, 150, 0.3)'  # Link color (translucent gray)
    }
))

# Adjust the layout
fig.update_layout(
    title="Three-level single-cell annotation Sankey diagram",
    font=dict(size=10),
    width=800,
    height=1000
)
fig.show()
fig.write_image("sankey_plot.pdf", engine="kaleido")

In [ ]:
# Save as a static image
fig.write_image("sankey_plot.pdf", engine="kaleido")  # Requires kaleido: pip install kaleido

## UMAP

In [ ]:
data = sc.read_h5ad('/path/to/hemelas_project/z_others/8_new/processed/adata.h5ad')
data

In [ ]:
subc = pd.read_csv('./processed/subcluster/celltype_l1_l3.csv', index_col=0)
subc

In [ ]:
data.obs['l3'] = subc['l3'].tolist()
data.obs['l1'] = subc['l1'].tolist()

In [ ]:
umap = np.load('/path/to/hemelas_project/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony_umap.npy')
umap.shape

In [ ]:
data.obsm['X_umap'] = umap

In [ ]:
sc.settings.set_figure_params(dpi=100, dpi_save=300, facecolor="white")

In [ ]:
import matplotlib.pyplot as plt
with plt.rc_context({"figure.figsize": (6, 6)}):
    sc.pl.umap(data, color='l3',size=0.2,save='l3_atac')

## Adjustment age

In [ ]:
data.obs['dataset'].value_counts()

In [ ]:
data.obs['age_stage'] = data.obs['age_stage'].tolist()[:data.shape[0]-53718]+ ['fetal']*53718

In [ ]:
import matplotlib.pyplot as plt
with plt.rc_context({"figure.figsize": (6, 6)}):
    sc.pl.umap(data, color='age_stage',size=0.2,save='age_stage')

## Main image drawing

In [ ]:
label = pd.DataFrame(['ELP','CMP','MEMP','HSC/MPP',
        'Regulatory T cells','Type 1 helper T cells','Follicular helper T cells','Tcm/Naive helper T cells','Tem/Trm cytotoxic T cells','Tcm/Naive cytotoxic T cells','Tem/Temra cytotoxic T cells',
        'ILC3','Transitional NK','CD16+ NK cells','Naive B cells','Memory B cells','Transitional B cells','Age-associated B cells',
        'Plasma cells','Plasmablasts', 'Pre-pro-B cells','Large pre-B cells','Mono-mac','Classical monocytes','Non-classical monocytes','DC1','DC2','DC3','DC precursor','Transitional DC',
        'Mid erythroid','Late erythroid','Early erythroid','Alveolar macrophages','Intermediate macrophages','Erythrophagocytic macrophages',
        'Cycling T cells','Cycling B cells','Cycling NK cells','pDC','Early MK','Mast cells'])

In [ ]:
label['num'] = range(1,43)
label

In [ ]:
label_dict = dict(zip(label[0], label['num']))
label_dict

In [ ]:
data.obs['label'] = data.obs['l3'].map(label_dict)

In [ ]:
sc.settings.set_figure_params(dpi=100, dpi_save=300, facecolor="white")

In [ ]:
import matplotlib.pyplot as plt
with plt.rc_context({"figure.figsize": (8, 8)}):
    sc.pl.umap(data, color='l3', )

In [ ]:
import matplotlib.pyplot as plt
with plt.rc_context({"figure.figsize": (8, 8)}):
    sc.pl.umap(data, color='label')

In [ ]:
data.obs['label'] = data.obs['label'].astype('category')

In [ ]:
df_sorted = label.sort_values(by=0)

In [ ]:
data

In [ ]:
df_sorted['color'] = data.uns['l3_colors']
df_sorted

In [ ]:
label = df_sorted.sort_values(by='num')
label

In [ ]:
data.uns['label_colors'] = label['color'].tolist()

In [ ]:
# main figure
import matplotlib.pyplot as plt
with plt.rc_context({"figure.figsize": (8, 8)}):
    sc.pl.umap(data, color='label',size=0.3,save='l3')